# Notebook: `02. Staging`
---
## Descrição

Esse notebook realiza o download sequencial dos arquivos `.parquet` da base de dados aberto da Taxi and Limousine Comission (TLC), e o download dos feriados federais e estaduais do estado de Nova Iorque.

---
## Origem
* Dados de viagens de taxi da TLC: Arquivos `.parquet` na estrutura de URL `https://d37ci6vzurychx.cloudfront.net/trip-data/{Dataset}_{Ano}-{Mes}.parquet`
* Dados de feriado: Consumo via biblioteca `holidays` do python

## Destino
1. Dados da TLC: Arquivo `.parquet` salvo em volume dedicado a cada tipo de dataset:
    - yellow_tripdata_ → `/Volumes/MVP/staging/yellow_taxi/`;
    - green_tripdata_ → `/Volumes/MVP/staging/green_taxi/`;
    - fhv_tripdata_ → `/Volumes/MVP/staging/fhv/`;
    - fhvhv_tripdata_ → `/Volumes/MVP/staging/fhvhv/`;

    **Modo de escrita**: Download incremental dos dados, usando o último valor Ano-Mês no nome do arquivo para fazer o download sequencial.

2. Dados de feriados: Arquivo `.csv` salvo no volume `/Volumes/mvp/staging/misc/`, sob o nome `Feriados_US_NY.csv`

    **Modo de escrita**: Sobrescreve o arquivo anterior a cada excecução.

Define o catálogo e schema a ser utilizado no notebook.

In [0]:
%sql
USE mvp.staging

Instalação das bibliotecas externas

In [0]:
%pip install holidays

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Importação das bibliotecas usadas

In [0]:
from datetime import date
from pathlib import Path
import pandas as pd
import re
import os
import time
import io
import requests
import holidays

## Definição das funções
---
**downloadArquivo**: Realiza o download do arquivo `.parquet` no URL informado e o salva no destino (volume) informado, com timeout de 60 segundos. Em caso de sucesso, emite uma mensagem informando o download do arquivo e retorna `True`. Caso o servidor retorne erro 404, informa que o arquivo não foi encontrado e retorna `False`, sem interromper a execução. Para qualquer outro erro HTTP, informa o código do erro no log e propaga a exceção.

**downloadTaxiData**: Para cada base configurada em `vehicleStartYear`, consulta `lastDownloaded` para identificar o último arquivo já baixado no volume. Caso não haja arquivos (`None`), inicia o download a partir do ano e mês de início definidos em `vehicleStartYear`. Caso já existam arquivos, retoma a partir do mês seguinte ao último baixado, avançando o ano quando necessário. Realiza o download sequencial mês a mês; ao encontrar um arquivo inexistente (HTTP 403 - Forbidden) para um determinado mês, interrompe o download daquele ano e avança para o ano seguinte da mesma base, encerrando no ano corrente da excecução.

**ultimoDownload**: Busca, no volume informado, o arquivo `.parquet` mais recente com base na ordenação do nome do arquivo, e extrai o ano e mês do padrão `YYYY-MM` presente no nome. Retorna `None` para ambas as variáveis caso o volume esteja vazio ou caso nenhum arquivo corresponda ao padrão esperado.

In [0]:
def downloadArquivo(url, destino):
    try:
        response = session.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)

        print(f"Baixado: {destino}")
        return True

    except requests.HTTPError as e:
        if e.response.status_code == 403:
            print(f"Não encontrado: {url}")
            return False
        else:
            print(f"Erro {e.response.status_code}: {url}")

        raise

def downloadDadosTaxis():
    anoCorrente = date.today().year
    for vehicle, (anoInicio, caminhoVolume, mesInicio) in veiculoAnoInicio.items():
        lastYear, lastMonth = ultimoDownload(caminhoVolume)
        if lastYear is None and lastMonth is None:
            anoInicioLoop = anoInicio
            mesInicioLoop = mesInicio
        else:
            anoInicioLoop = lastYear
            mesInicioLoop = lastMonth + 1

            if mesInicioLoop == 13:
                mesInicioLoop = 1
                anoInicioLoop += 1

        for year in range(anoInicioLoop, anoCorrente + 1):

            firstMonth =(
                mesInicioLoop
                if year == anoInicioLoop
                else 1
            )

            for month in range(firstMonth, 13):
                nomeArquivo = f"{vehicle}{year}-{month:02d}.parquet"
                url = f"{urlRaiz}{nomeArquivo}"
                destino = os.path.join(
                    caminhoVolume,
                    nomeArquivo
                )

                success = downloadArquivo(url, destino)

                time.sleep(0.25)

                if not success:
                    break

def ultimoDownload(caminhoVolume):
    files = list(Path(caminhoVolume).glob("*.parquet"))
    if not files:
        return None, None
    
    latest = max(files, key=lambda x: x.name)
    match = re.search(r'(\d{4})-(\d{2})', latest.name)

    if not match:
        return None, None

    year, month = map(int, match.groups())

    return year, month

Bloco de configuração e execução: 
* Define `urlRaiz`, a URL raiz do repositório de dados da TLC; 
* `vehicleStartYear`, dicionário com o prefixo do nome de arquivo, ano de início, caminho do volume de destino e mês de início para cada base de dados (yellow, green, fhv e fhvhv); 
* `session`, um objeto `requests.Session` reutilizado pelas chamadas HTTP de `downloadFile`. 

Essas variáveis são lidas globalmente por `downloadTaxiData`, que é então chamada para iniciar o download sequencial dos arquivos `.parquet` de todas as bases configuradas.

In [0]:
urlRaiz = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

veiculoAnoInicio = {
    'yellow_tripdata_': [2016, '/Volumes/MVP/staging/yellow_taxi/', 1],
    'green_tripdata_': [2016, '/Volumes/MVP/staging/green_taxi/', 1],
    'fhv_tripdata_': [2016, '/Volumes/MVP/staging/fhv/', 1],
    'fhvhv_tripdata_': [2019, '/Volumes/MVP/staging/fhvhv/', 2]
}

session = requests.Session()

downloadDadosTaxis()

Não encontrado: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-08.parquet
Não encontrado: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-08.parquet
Não encontrado: https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2026-07.parquet
Não encontrado: https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-08.parquet


Gera e salva os dados de feriados no volume `misc` da camada `staging`.

In [0]:
us_ny = holidays.US(state="NY", years=range(2016, 2027))
df = pd.DataFrame(list(us_ny.items()), columns=["date", "holiday_name"])

csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)
dbutils.fs.put("/Volumes/mvp/staging/misc/Feriados_US_NY.csv", csv_buffer.getvalue(), overwrite=True)

Wrote 4895 bytes.


True